# SE-BRL Data Cleaning

## Purpose

This notebook performs reproducible data-cleaning procedures for the raw
datasets used in the SE-BRL experimental pipeline.

Cleaning decisions are derived from findings established in
`01_data_audit.ipynb`. The objective is to correct or remove documented
data-quality issues while preserving the original source information and
avoiding subjective or human-assigned annotations.

## Cleaning Principles

The following principles are applied throughout this notebook:

- Raw source files under `experiments/datasets-raw/` remain immutable.
- Every cleaning operation must be justified by an audit finding.
- No manual phishing or behavioral annotation is performed.
- Native labels are preserved until a source-supported mapping is defined.
- Spam is not automatically treated as equivalent to phishing.
- Cleaning decisions are applied programmatically and reproducibly.
- Before-and-after record counts are recorded for every operation.
- Potential grouping information required for leakage prevention is preserved.
- Model-dependent transformations are deferred to preprocessing.
- Development, calibration, and test partitions are not created or accessed
  during data cleaning.

## Inputs

Raw datasets:

`experiments/datasets-raw/`

Audit evidence:

`experiments/results/audits/`

## Outputs

Cleaned intermediate datasets will be written to:

`experiments/data/interim/`

A cleaning log will record the operations performed and their effects on
each dataset.

## Workflow

This notebook will:

1. load the audited raw datasets;
2. establish dataset-specific cleaning rules from the audit findings;
3. handle exact duplicate observations where justified;
4. address documented missing and non-finite values;
5. investigate structurally extreme email observations;
6. standardize representations required for reliable downstream processing;
7. preserve identifiers required for leakage-aware grouping;
8. validate cleaned outputs against explicit integrity checks; and
9. export cleaned intermediate datasets and cleaning documentation.

Feature scaling, model-specific encoding, train/calibration/test partitioning,
and SE-BRL behavioral feature extraction are outside the scope of this
notebook.

## 1. Environment and Paths

The cleaning environment is initialized using the same repository-relative
path structure established during the data audit. Raw datasets are loaded
from the immutable source directory, while cleaned outputs are written
separately to the intermediate-data directory.

In [24]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import arff

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

In [25]:
# Locate repository root

cwd = Path.cwd().resolve()

REPO_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / ".git").exists()
        and (p / "experiments").exists()
    ),
    None
)

if REPO_ROOT is None:
    raise RuntimeError(
        "Could not locate repository root."
    )

RAW_DATA_DIR = REPO_ROOT / "experiments" / "datasets-raw"
AUDIT_DIR = REPO_ROOT / "experiments" / "results" / "audits"
INTERIM_DIR = REPO_ROOT / "experiments" / "data" / "interim"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Raw data:", RAW_DATA_DIR)
print("Audit results:", AUDIT_DIR)
print("Cleaned output:", INTERIM_DIR)

FileNotFoundError: [Errno 2] No such file or directory

In [ ]:
print("Raw data exists:", RAW_DATA_DIR.exists())
print("Audit results exist:", AUDIT_DIR.exists())
print("Interim output exists:", INTERIM_DIR.exists())

Raw data exists: True
Audit results exist: True
Interim output exists: True


## 2. Load Audit Evidence

The cleaning procedures in this notebook are based on the findings produced
during the dataset audit. The exported audit reports are loaded before any
cleaning operation is performed so that each decision can be traced to a
documented data-quality issue.

These reports are treated as evidence only and are not modified during the
cleaning process.

In [ ]:
audit_files = {
    "inventory": "dataset_inventory.csv",
    "labels": "label_distribution.csv",
    "label_semantics": "verified_label_semantics.csv",
    "missing": "missing_values.csv",
    "duplicates": "exact_duplicates.csv",
    "email_stats": "email_content_statistics.csv",
    "email_lengths": "email_length_distribution.csv",
    "numeric": "structured_numeric_audit.csv",
    "uci_values": "uci_native_value_audit.csv",
    "phiusiil_grouping": "phiusiil_grouping_summary.csv",
    "phiusiil_labels": "phiusiil_domain_label_summary.csv",
    "email_grouping": "email_grouping_summary.csv",
    "sender_labels": "email_sender_label_summary.csv",
}

audit_reports = {
    name: pd.read_csv(AUDIT_DIR / filename)
    for name, filename in audit_files.items()
}

print("Audit reports loaded:", len(audit_reports))

for name, report in audit_reports.items():
    print(f"{name:24s} {report.shape}")

Audit reports loaded: 13
inventory                (8, 8)
labels                   (16, 5)
label_semantics          (9, 4)
missing                  (271, 4)
duplicates               (8, 4)
email_stats              (4, 11)
email_lengths            (4, 9)
numeric                  (130, 9)
uci_values               (31, 5)
phiusiil_grouping        (1, 7)
phiusiil_labels          (1, 3)
email_grouping           (4, 8)
sender_labels            (3, 4)


## 3. Cleaning Decision Log

A cleaning log is maintained to document every modification applied to the
datasets. Each entry records the dataset, observed issue, audit evidence,
cleaning action, rationale, and resulting change in record count.

This provides a reproducible record of the cleaning process and distinguishes
actual data modifications from issues intentionally deferred to later stages,
such as model-dependent imputation, feature selection, and scaling.

In [ ]:
cleaning_log = []

def log_cleaning(
    dataset,
    issue,
    evidence,
    action,
    rationale,
    rows_before,
    rows_after
):
    cleaning_log.append({
        "dataset": dataset,
        "issue": issue,
        "audit_evidence": evidence,
        "action": action,
        "rationale": rationale,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after
    })

print("Cleaning log initialized.")

Cleaning log initialized.


## 4. Load Raw Datasets

Fresh copies of the audited source datasets are loaded directly from the
immutable raw-data directory. Loading the data again ensures that cleaning
begins from the original source files rather than from objects that may have
been modified during the audit.

The datasets are loaded without changing labels, removing observations,
imputing missing values, or transforming features.

In [ ]:
DATASETS = {
    "CEAS-08": {
        "path": RAW_DATA_DIR / "email" / "ceas_08" / "CEAS_08.csv",
        "format": "csv"
    },
    "Enron": {
        "path": RAW_DATA_DIR / "email" / "enron" / "Enron.csv",
        "format": "csv"
    },
    "Nazario": {
        "path": RAW_DATA_DIR / "email" / "nazario" / "Nazario.csv",
        "format": "csv"
    },
    "SpamAssassin": {
        "path": RAW_DATA_DIR / "email" / "spamassasin" / "SpamAssasin.csv",
        "format": "csv"
    },
    "PhiUSIIL": {
        "path": RAW_DATA_DIR / "webpage" / "PhiUSIL"
                / "PhiUSIIL_Phishing_URL_Dataset.csv",
        "format": "csv"
    },
    "UCI Phishing Websites": {
        "path": RAW_DATA_DIR / "webpage" / "uci"
                / "phishing+websites" / "Training Dataset.arff",
        "format": "arff"
    },
    "ISCX-URL2016 All": {
        "path": RAW_DATA_DIR / "url" / "iscx-url-2016" / "All.csv",
        "format": "csv"
    },
    "ISCX-URL2016 Phishing": {
        "path": RAW_DATA_DIR / "url" / "iscx-url-2016" / "Phishing.csv",
        "format": "csv"
    },
}

In [ ]:
def load_dataset(info):
    if info["format"] == "csv":
        return pd.read_csv(info["path"])

    if info["format"] == "arff":
        data, _ = arff.loadarff(info["path"])
        return pd.DataFrame(data)

    raise ValueError(f"Unsupported format: {info['format']}")


raw_data = {
    name: load_dataset(info)
    for name, info in DATASETS.items()
}

print("Raw datasets loaded:", len(raw_data))

for name, df in raw_data.items():
    print(f"{name:26s} {df.shape}")

Raw datasets loaded: 8
CEAS-08                    (39154, 7)
Enron                      (29767, 3)
Nazario                    (1565, 7)
SpamAssassin               (5809, 7)
PhiUSIIL                   (235795, 56)
UCI Phishing Websites      (11055, 31)
ISCX-URL2016 All           (36707, 80)
ISCX-URL2016 Phishing      (15367, 80)


## 5. Pre-Cleaning Integrity Verification

Before applying any cleaning operation, the freshly loaded datasets are
compared with the dimensions recorded during the audit.

This check confirms that the cleaning stage is operating on the same source
files examined in `01_data_audit.ipynb` and establishes the baseline record
counts used to measure subsequent cleaning effects.

In [ ]:
baseline_shapes = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(df),
        "columns": len(df.columns)
    }
    for name, df in raw_data.items()
])

baseline_shapes

,dataset,rows,columns
0,CEAS-08,39154,7
1,Enron,29767,3
2,Nazario,1565,7
3,SpamAssassin,5809,7
4,PhiUSIIL,235795,56
5,UCI Phishing Websites,11055,31
6,ISCX-URL2016 All,36707,80
7,ISCX-URL2016 Phishing,15367,80


In [ ]:
clean_data = {
    name: df.copy(deep=True)
    for name, df in raw_data.items()
}

print("Working copies created:", len(clean_data))

Working copies created: 8


## 6. Exact Duplicate Resolution

The audit identified substantial exact duplication in the UCI Phishing
Websites and ISCX-URL2016 datasets. Exact duplicate records can artificially
increase the influence of repeated observations and may contribute to
information leakage if identical records are distributed across experimental
partitions.

Before removing duplicates, predictor-level duplicates are examined for
label conflicts. This distinguishes genuinely repeated observations from
cases where identical predictor values are associated with different class
labels.

Only complete duplicate rows are removed during this stage. Repeated URLs,
domains, subjects, or other partial matches are not treated as exact
duplicates and are preserved for later leakage-aware grouping.

In [ ]:
duplicate_checks = []

duplicate_targets = {
    "UCI Phishing Websites": "Result",
    "ISCX-URL2016 All": "URL_Type_obf_Type",
    "ISCX-URL2016 Phishing": "URL_Type_obf_Type",
}

for name, label_col in duplicate_targets.items():
    df = clean_data[name]

    feature_cols = [
        col for col in df.columns
        if col != label_col
    ]

    exact_duplicates = df.duplicated(keep="first").sum()

    feature_label_counts = (
        df.groupby(feature_cols, dropna=False)[label_col]
        .nunique()
    )

    conflicting_groups = (feature_label_counts > 1).sum()

    duplicate_checks.append({
        "dataset": name,
        "rows": len(df),
        "exact_duplicates": int(exact_duplicates),
        "conflicting_feature_groups": int(conflicting_groups)
    })

duplicate_checks = pd.DataFrame(duplicate_checks)

duplicate_checks

,dataset,rows,exact_duplicates,conflicting_feature_groups
0,UCI Phishing Websites,11055,5206,64
1,ISCX-URL2016 All,36707,9754,0
2,ISCX-URL2016 Phishing,15367,544,0


### 6.1 Remove Complete Duplicate Rows

Predictor-level analysis identified 64 conflicting feature groups in the UCI
dataset, where identical predictor values are associated with different native
labels. These observations are preserved because resolving their labels would
require an unsupported labeling decision.

No predictor-level label conflicts were detected in the ISCX-URL2016 files.

Accordingly, duplicate removal is restricted to complete row duplicates,
including the native label. This removes only observations that are identical
across all available fields while preserving conflicting predictor-label
combinations.

In [ ]:
for name in duplicate_targets:
    df = clean_data[name]

    rows_before = len(df)

    clean_data[name] = (
        df.drop_duplicates()
        .reset_index(drop=True)
    )

    rows_after = len(clean_data[name])

    log_cleaning(
        dataset=name,
        issue="Exact duplicate rows",
        evidence="exact_duplicates.csv and predictor-label conflict check",
        action="Removed complete duplicate rows only",
        rationale=(
            "Exact copies provide no additional unique information. "
            "Predictor-level duplicates with different labels were preserved."
        ),
        rows_before=rows_before,
        rows_after=rows_after
    )

    print(
        f"{name}: {rows_before:,} -> {rows_after:,} "
        f"({rows_before - rows_after:,} removed)"
    )

UCI Phishing Websites: 11,055 -> 5,849 (5,206 removed)
ISCX-URL2016 All: 36,707 -> 26,953 (9,754 removed)
ISCX-URL2016 Phishing: 15,367 -> 14,823 (544 removed)


In [ ]:
pd.DataFrame(cleaning_log)

,dataset,issue,audit_evidence,action,rationale,rows_before,rows_after,rows_removed
0,UCI Phishing Websites,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,11055,5849,5206
1,ISCX-URL2016 All,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,36707,26953,9754
2,ISCX-URL2016 Phishing,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,15367,14823,544


## 7. Missing-Value Handling

Missing values are handled according to their role and the findings of the
data audit. Deterministic corrections may be performed during cleaning, while
statistical or model-dependent imputation is deferred to preprocessing to
prevent information leakage.

### 7.1 Missing Email Text Fields

The audit found missing values in some email subject and body fields, but no
email observation had both fields missing simultaneously. Therefore, these
records remain usable and are not removed.

Missing subject or body values are replaced with empty strings. This preserves
the available textual content while providing a consistent representation for
downstream text processing. Other metadata fields are not imputed at this
stage unless required by a documented cleaning rule.

In [ ]:
email_datasets = [
    "CEAS-08",
    "Enron",
    "Nazario",
    "SpamAssassin"
]

for name in email_datasets:
    df = clean_data[name]

    text_cols = [
        col for col in ["subject", "body"]
        if col in df.columns
    ]

    missing_before = int(df[text_cols].isna().sum().sum())

    df[text_cols] = df[text_cols].fillna("")

    missing_after = int(df[text_cols].isna().sum().sum())

    print(
        f"{name}: {missing_before} -> "
        f"{missing_after} missing text values"
    )

CEAS-08: 28 -> 0 missing text values
Enron: 198 -> 0 missing text values
Nazario: 4 -> 0 missing text values
SpamAssassin: 17 -> 0 missing text values


### 7.2 Record Email Text Cleaning

The missing-text replacement affected 28 values in CEAS-08, 198 in Enron,
4 in Nazario, and 17 in SpamAssassin. No rows were removed because each
affected record retained usable email content.

These counts refer specifically to missing values in the `subject` and `body`
fields and therefore may differ from dataset-wide missing-value totals reported
during the audit.

In [ ]:
text_missing_counts = {
    "CEAS-08": 28,
    "Enron": 198,
    "Nazario": 4,
    "SpamAssassin": 17
}

for name, affected in text_missing_counts.items():
    cleaning_log.append({
        "dataset": name,
        "issue": "Missing email text",
        "audit_evidence": "missing_values.csv and email content audit",
        "action": "Replaced missing subject/body values with empty strings",
        "rationale": (
            "No email had both subject and body absent, so usable records "
            "were preserved without statistical imputation."
        ),
        "rows_before": len(clean_data[name]),
        "rows_after": len(clean_data[name]),
        "rows_removed": 0,
        "values_modified": affected
    })

pd.DataFrame(cleaning_log)

,dataset,issue,audit_evidence,action,rationale,rows_before,rows_after,rows_removed,values_modified
0,UCI Phishing Websites,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,11055,5849,5206,NaN
1,ISCX-URL2016 All,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,36707,26953,9754,NaN
2,ISCX-URL2016 Phishing,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,15367,14823,544,NaN
3,CEAS-08,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",39154,39154,0,28.0
4,Enron,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",29767,29767,0,198.0
5,Nazario,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",1565,1565,0,4.0
6,SpamAssassin,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",5809,5809,0,17.0


## 8. Non-Finite Structured Values

The audit identified missing and infinite numeric values in several
ISCX-URL2016 features. These values require separate treatment from ordinary
email-text missingness.

Statistical imputation is not performed during cleaning because parameters
derived from the complete dataset could introduce information leakage.
Instead, this stage first identifies non-finite values that require
deterministic sanitization. Model-dependent imputation will be performed
later using development or fold-local information.

Values such as `-1` are preserved unless source documentation establishes
that they represent invalid or missing data.

In [ ]:
for name in [
    "ISCX-URL2016 All",
    "ISCX-URL2016 Phishing"
]:
    df = clean_data[name]

    numeric_cols = df.select_dtypes(include=np.number).columns

    missing_count = int(df[numeric_cols].isna().sum().sum())

    inf_count = int(
        np.isinf(df[numeric_cols].to_numpy(dtype=float)).sum()
    )

    print(
        f"{name}: "
        f"{missing_count:,} missing values, "
        f"{inf_count:,} infinite values"
    )

ISCX-URL2016 All: 14,723 missing values, 10 infinite values
ISCX-URL2016 Phishing: 9,321 missing values, 9 infinite values


### 8.1 Standardize Infinite Values

After exact duplicate removal, the ISCX-URL2016 datasets contained 14,723
missing and 10 infinite numeric values in `All.csv`, and 9,321 missing and
9 infinite numeric values in `Phishing.csv`.

Infinite values cannot be used directly by the downstream machine-learning
models. They are therefore standardized to `NaN`, allowing all unavailable
or non-finite numeric values to be represented consistently.

No statistical imputation is performed during cleaning. Missing values will
be handled later using leakage-safe preprocessing procedures. Existing `-1`
values are also preserved because they are not assumed to represent missing
data without source-supported evidence.

In [ ]:
for name in [
    "ISCX-URL2016 All",
    "ISCX-URL2016 Phishing"
]:
    df = clean_data[name]

    numeric_cols = df.select_dtypes(include=np.number).columns

    inf_before = int(
        np.isinf(df[numeric_cols].to_numpy(dtype=float)).sum()
    )

    clean_data[name][numeric_cols] = (
        df[numeric_cols].replace([np.inf, -np.inf], np.nan)
    )

    inf_after = int(
        np.isinf(
            clean_data[name][numeric_cols].to_numpy(dtype=float)
        ).sum()
    )

    missing_after = int(
        clean_data[name][numeric_cols].isna().sum().sum()
    )

    print(
        f"{name}: {inf_before} -> {inf_after} infinite values; "
        f"{missing_after:,} missing values after standardization"
    )

ISCX-URL2016 All: 10 -> 0 infinite values; 14,733 missing values after standardization
ISCX-URL2016 Phishing: 9 -> 0 infinite values; 9,330 missing values after standardization


In [ ]:
iscx_inf_counts = {
    "ISCX-URL2016 All": 10,
    "ISCX-URL2016 Phishing": 9
}

for name, affected in iscx_inf_counts.items():
    cleaning_log.append({
        "dataset": name,
        "issue": "Infinite numeric values",
        "audit_evidence": "structured_numeric_audit.csv and post-deduplication check",
        "action": "Converted positive and negative infinity to NaN",
        "rationale": (
            "Non-finite values cannot be used directly by downstream models. "
            "They were standardized as missing values without performing "
            "statistical imputation."
        ),
        "rows_before": len(clean_data[name]),
        "rows_after": len(clean_data[name]),
        "rows_removed": 0,
        "values_modified": affected
    })

pd.DataFrame(cleaning_log)

,dataset,issue,audit_evidence,action,rationale,rows_before,rows_after,rows_removed,values_modified
0,UCI Phishing Websites,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,11055,5849,5206,NaN
1,ISCX-URL2016 All,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,36707,26953,9754,NaN
2,ISCX-URL2016 Phishing,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,15367,14823,544,NaN
3,CEAS-08,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",39154,39154,0,28.0
4,Enron,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",29767,29767,0,198.0
5,Nazario,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",1565,1565,0,4.0
6,SpamAssassin,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",5809,5809,0,17.0
7,ISCX-URL2016 All,Infinite numeric values,structured_numeric_audit.csv and post-deduplication check,Converted positive and negative infinity to NaN,Non-finite values cannot be used directly by downstream models. They were standardized as missin...,26953,26953,0,10.0
8,ISCX-URL2016 Phishing,Infinite numeric values,structured_numeric_audit.csv and post-deduplication check,Converted positive and negative infinity to NaN,Non-finite values cannot be used directly by downstream models. They were standardized as missin...,14823,14823,0,9.0


## 9. UCI Native Value Standardization

The UCI Phishing Websites dataset is distributed in ARFF format. When loaded
through SciPy, its categorical numeric values are represented as byte strings,
such as `b'-1'`, `b'0'`, and `b'1'`.

These values are converted to their corresponding integer representations.
This is a representation-only conversion and does not alter the native feature
or label semantics.

The native `Result` values are preserved as `-1` and `1`; no harmonized
phishing label is assigned during this cleaning step.

In [ ]:
uci = clean_data["UCI Phishing Websites"]

for col in uci.columns:
    uci[col] = uci[col].apply(
        lambda x: int(x.decode()) if isinstance(x, bytes) else int(x)
    )

print("Data types:")
print(uci.dtypes.value_counts())

print("\nResult values:")
print(sorted(uci["Result"].unique()))

Data types:
int64    31
Name: count, dtype: int64

Result values:
[np.int64(-1), np.int64(1)]


In [ ]:
cleaning_log.append({
    "dataset": "UCI Phishing Websites",
    "issue": "ARFF byte-string representation",
    "audit_evidence": "uci_native_value_audit.csv",
    "action": "Converted byte-string values to integers",
    "rationale": (
        "The ARFF loader represented native values as byte strings. "
        "Conversion to integers standardizes their representation without "
        "changing feature or label semantics."
    ),
    "rows_before": len(clean_data["UCI Phishing Websites"]),
    "rows_after": len(clean_data["UCI Phishing Websites"]),
    "rows_removed": 0,
    "values_modified": len(clean_data["UCI Phishing Websites"]) * 31
})

pd.DataFrame(cleaning_log)

,dataset,issue,audit_evidence,action,rationale,rows_before,rows_after,rows_removed,values_modified
0,UCI Phishing Websites,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,11055,5849,5206,NaN
1,ISCX-URL2016 All,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,36707,26953,9754,NaN
2,ISCX-URL2016 Phishing,Exact duplicate rows,exact_duplicates.csv and predictor-label conflict check,Removed complete duplicate rows only,Exact copies provide no additional unique information. Predictor-level duplicates with different...,15367,14823,544,NaN
3,CEAS-08,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",39154,39154,0,28.0
4,Enron,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",29767,29767,0,198.0
5,Nazario,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",1565,1565,0,4.0
6,SpamAssassin,Missing email text,missing_values.csv and email content audit,Replaced missing subject/body values with empty strings,"No email had both subject and body absent, so usable records were preserved without statistical ...",5809,5809,0,17.0
7,ISCX-URL2016 All,Infinite numeric values,structured_numeric_audit.csv and post-deduplication check,Converted positive and negative infinity to NaN,Non-finite values cannot be used directly by downstream models. They were standardized as missin...,26953,26953,0,10.0
8,ISCX-URL2016 Phishing,Infinite numeric values,structured_numeric_audit.csv and post-deduplication check,Converted positive and negative infinity to NaN,Non-finite values cannot be used directly by downstream models. They were standardized as missin...,14823,14823,0,9.0
9,UCI Phishing Websites,ARFF byte-string representation,uci_native_value_audit.csv,Converted byte-string values to integers,The ARFF loader represented native values as byte strings. Conversion to integers standardizes t...,5849,5849,0,181319.0


## 10. Extreme Email-Length Review

The audit identified a small number of unusually long email bodies, including
extreme observations in the Nazario dataset.

Email length alone is not sufficient evidence that an observation is invalid
or corrupted. Removing records solely because they exceed an arbitrary length
threshold could discard legitimate source observations and alter the dataset
distribution.

Therefore, extreme-length records are retained during cleaning unless a
reproducible structural check identifies an invalid or unusable observation.
Length restrictions required by downstream feature extraction or modeling are
deferred to preprocessing.

In [26]:
extreme_email_summary = []

for name in email_datasets:
    df = clean_data[name]

    body_length = df["body"].astype(str).str.len()

    extreme_email_summary.append({
        "dataset": name,
        "max_body_length": int(body_length.max()),
        "over_100k": int((body_length > 100_000).sum()),
        "over_1m": int((body_length > 1_000_000).sum())
    })

pd.DataFrame(extreme_email_summary)

,dataset,max_body_length,over_100k,over_1m
0,CEAS-08,143996,2,0
1,Enron,228353,10,0
2,Nazario,4599644,2,2
3,SpamAssassin,299201,6,0


### 10.1 Extreme-Length Decision

The structural review confirmed that extreme email bodies are rare. CEAS-08,
Enron, Nazario, and SpamAssassin contain 2, 10, 2, and 6 bodies above 100,000
characters, respectively. Only Nazario contains observations above one million
characters, with two such records and a maximum body length of 4,599,644
characters.

No records are removed on the basis of length alone. The observed values are
treated as valid source observations because the audit provides no objective
evidence that they represent corrupted records.

Any truncation or length restriction required by downstream models will be
defined and applied consistently during preprocessing rather than data
cleaning.

In [27]:
for row in extreme_email_summary:
    name = row["dataset"]

    cleaning_log.append({
        "dataset": name,
        "issue": "Extreme email body length",
        "audit_evidence": "email_length_distribution.csv and structural length review",
        "action": "Retained extreme-length observations",
        "rationale": (
            "Length alone does not establish corruption. No objective "
            "cleaning criterion justified removing these source records."
        ),
        "rows_before": len(clean_data[name]),
        "rows_after": len(clean_data[name]),
        "rows_removed": 0,
        "values_modified": 0
    })

pd.DataFrame(cleaning_log).tail(4)

,dataset,issue,audit_evidence,action,rationale,rows_before,rows_after,rows_removed,values_modified
10,CEAS-08,Extreme email body length,email_length_distribution.csv and structural length review,Retained extreme-length observations,Length alone does not establish corruption. No objective cleaning criterion justified removing t...,39154,39154,0,0.0
11,Enron,Extreme email body length,email_length_distribution.csv and structural length review,Retained extreme-length observations,Length alone does not establish corruption. No objective cleaning criterion justified removing t...,29767,29767,0,0.0
12,Nazario,Extreme email body length,email_length_distribution.csv and structural length review,Retained extreme-length observations,Length alone does not establish corruption. No objective cleaning criterion justified removing t...,1565,1565,0,0.0
13,SpamAssassin,Extreme email body length,email_length_distribution.csv and structural length review,Retained extreme-length observations,Length alone does not establish corruption. No objective cleaning criterion justified removing t...,5809,5809,0,0.0


## 11. Native Label Semantics Review

Class labels are reviewed before any harmonization is performed. The source
datasets use different labeling schemes, and semantically different classes
must not be treated as interchangeable.

In particular, spam or unsolicited email is not automatically equivalent to
phishing. Assigning a phishing label without source-supported provenance would
introduce an unsupported annotation and violate the study's no-human-annotation
constraint.

Therefore, native labels are preserved during cleaning. Label harmonization
is performed only when the class meaning is explicitly supported by source
documentation and required for downstream experimental construction.

In [28]:
label_semantics = audit_reports["label_semantics"].copy()

label_semantics

,dataset,native_label,verified_meaning,status
0,PhiUSIIL,1,legitimate,verified from source documentation
1,PhiUSIIL,0,phishing,verified from source documentation
2,UCI Phishing Websites,1,legitimate,verified from source documentation
3,UCI Phishing Websites,-1,phishy,verified from source documentation
4,ISCX-URL2016,native strings,benign / phishing / spam / malware / Defacement,explicit in raw dataset
5,CEAS-08,0 / 1,pending provenance verification,do not map yet
6,Enron,0 / 1,pending provenance verification,do not map yet
7,Nazario,1,pending provenance verification,do not map yet
8,SpamAssassin,0 / 1,pending provenance verification,do not map yet


In [29]:
for name, df in clean_data.items():
    if name in ["CEAS-08", "Enron", "Nazario", "SpamAssassin"]:
        label_col = "label"
    elif name == "PhiUSIIL":
        label_col = "label"
    elif name == "UCI Phishing Websites":
        label_col = "Result"
    elif name in ["ISCX-URL2016 All", "ISCX-URL2016 Phishing"]:
        label_col = "URL_Type_obf_Type"
    else:
        continue

    print(f"\n{name}")
    print(df[label_col].value_counts(dropna=False))


CEAS-08
label
1    21842
0    17312
Name: count, dtype: int64

Enron
label
0    15791
1    13976
Name: count, dtype: int64

Nazario
label
1    1565
Name: count, dtype: int64

SpamAssassin
label
0    4091
1    1718
Name: count, dtype: int64

PhiUSIIL
label
1    134850
0    100945
Name: count, dtype: int64

UCI Phishing Websites
Result
-1    3019
 1    2830
Name: count, dtype: int64

ISCX-URL2016 All
URL_Type_obf_Type
benign        7464
phishing      7359
spam          5331
Defacement    5068
malware       1731
Name: count, dtype: int64

ISCX-URL2016 Phishing
URL_Type_obf_Type
benign      7464
phishing    7359
Name: count, dtype: int64


### 11.1 Label-Preservation Decision

Post-cleaning verification confirmed that the source-native class labels remain
intact. Changes in class counts for the UCI Phishing Websites and ISCX-URL2016
datasets are attributable only to the removal of complete duplicate rows.

No class labels are harmonized during data cleaning. This decision prevents
semantically different source classes, particularly spam and phishing, from
being treated as equivalent without explicit source-supported justification.

The cleaned datasets therefore retain their native labels. Any target-label
mapping required by the SE-BRL experiments will be defined explicitly during
downstream experimental construction using documented dataset semantics.

In [30]:
for name in clean_data:
    cleaning_log.append({
        "dataset": name,
        "issue": "Native label semantics",
        "audit_evidence": "label_distribution.csv and verified_label_semantics.csv",
        "action": "Preserved native labels without harmonization",
        "rationale": (
            "Cleaning must not introduce unsupported class mappings. "
            "Spam and phishing are not assumed to be equivalent, and "
            "experimental target construction is deferred to a later stage."
        ),
        "rows_before": len(clean_data[name]),
        "rows_after": len(clean_data[name]),
        "rows_removed": 0,
        "values_modified": 0
    })

pd.DataFrame(cleaning_log).tail(8)

,dataset,issue,audit_evidence,action,rationale,rows_before,rows_after,rows_removed,values_modified
14,CEAS-08,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,39154,39154,0,0.0
15,Enron,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,29767,29767,0,0.0
16,Nazario,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,1565,1565,0,0.0
17,SpamAssassin,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,5809,5809,0,0.0
18,PhiUSIIL,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,235795,235795,0,0.0
19,UCI Phishing Websites,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,5849,5849,0,0.0
20,ISCX-URL2016 All,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,26953,26953,0,0.0
21,ISCX-URL2016 Phishing,Native label semantics,label_distribution.csv and verified_label_semantics.csv,Preserved native labels without harmonization,Cleaning must not introduce unsupported class mappings. Spam and phishing are not assumed to be ...,14823,14823,0,0.0


## 12. Post-Cleaning Validation

The cleaned datasets are validated before export to ensure that the documented
cleaning operations produced the intended results without introducing
unexpected changes.

The validation checks confirm that:

- complete duplicate rows have been removed from datasets where deduplication
  was required;
- email subject and body fields no longer contain missing values;
- ISCX-URL2016 numeric features contain no infinite values;
- UCI ARFF byte-string values have been converted to integer representations;
- native class labels remain available; and
- all datasets retain at least one observation.

Missing numeric values intentionally deferred to leakage-safe preprocessing
are permitted at this stage.

In [31]:
validation_results = []

# 1. Exact duplicates
for name in [
    "UCI Phishing Websites",
    "ISCX-URL2016 All",
    "ISCX-URL2016 Phishing"
]:
    remaining = int(clean_data[name].duplicated().sum())

    validation_results.append({
        "dataset": name,
        "check": "No complete duplicate rows",
        "passed": remaining == 0,
        "details": f"{remaining} duplicates remaining"
    })


# 2. Missing email text
for name in email_datasets:
    text_cols = [
        col for col in ["subject", "body"]
        if col in clean_data[name].columns
    ]

    remaining = int(
        clean_data[name][text_cols].isna().sum().sum()
    )

    validation_results.append({
        "dataset": name,
        "check": "No missing subject/body values",
        "passed": remaining == 0,
        "details": f"{remaining} missing text values"
    })


# 3. ISCX infinite values
for name in [
    "ISCX-URL2016 All",
    "ISCX-URL2016 Phishing"
]:
    numeric_cols = clean_data[name].select_dtypes(
        include=np.number
    ).columns

    remaining = int(
        np.isinf(
            clean_data[name][numeric_cols].to_numpy(dtype=float)
        ).sum()
    )

    validation_results.append({
        "dataset": name,
        "check": "No infinite numeric values",
        "passed": remaining == 0,
        "details": f"{remaining} infinite values"
    })


# 4. UCI representation
uci = clean_data["UCI Phishing Websites"]

uci_all_integer = all(
    pd.api.types.is_integer_dtype(uci[col])
    for col in uci.columns
)

validation_results.append({
    "dataset": "UCI Phishing Websites",
    "check": "UCI values converted to integer representation",
    "passed": uci_all_integer,
    "details": f"{len(uci.columns)} columns checked"
})


# 5. Non-empty datasets
for name, df in clean_data.items():
    validation_results.append({
        "dataset": name,
        "check": "Dataset contains observations",
        "passed": len(df) > 0,
        "details": f"{len(df):,} rows"
    })


validation_results = pd.DataFrame(validation_results)

validation_results

,dataset,check,passed,details
0,UCI Phishing Websites,No complete duplicate rows,True,0 duplicates remaining
1,ISCX-URL2016 All,No complete duplicate rows,True,0 duplicates remaining
2,ISCX-URL2016 Phishing,No complete duplicate rows,True,0 duplicates remaining
3,CEAS-08,No missing subject/body values,True,0 missing text values
4,Enron,No missing subject/body values,True,0 missing text values
5,Nazario,No missing subject/body values,True,0 missing text values
6,SpamAssassin,No missing subject/body values,True,0 missing text values
7,ISCX-URL2016 All,No infinite numeric values,True,0 infinite values
8,ISCX-URL2016 Phishing,No infinite numeric values,True,0 infinite values
9,UCI Phishing Websites,UCI values converted to integer representation,True,31 columns checked


In [32]:
failed_checks = validation_results[
    ~validation_results["passed"]
]

if len(failed_checks) == 0:
    print(
        f"All {len(validation_results)} "
        "post-cleaning validation checks passed."
    )
else:
    print("Validation failures detected:")
    display(failed_checks)

All 18 post-cleaning validation checks passed.


## 13. Export Cleaned Data and Documentation

After successful post-cleaning validation, the cleaned datasets are exported
to the intermediate-data directory for use by the preprocessing stage.

The cleaning decision log and validation results are also exported separately
to preserve a reproducible record of the operations performed in this
notebook.

Raw source files remain unchanged. The exported datasets retain their native
labels and any missing numeric values intentionally deferred to leakage-safe
preprocessing.

In [33]:
CLEANING_RESULTS_DIR = REPO_ROOT / "experiments" / "results" / "cleaning"

EMAIL_OUT = INTERIM_DIR / "email"
WEBPAGE_OUT = INTERIM_DIR / "webpage"
URL_OUT = INTERIM_DIR / "url"

for directory in [
    CLEANING_RESULTS_DIR,
    EMAIL_OUT,
    WEBPAGE_OUT,
    URL_OUT
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Output directories ready.")

Output directories ready.


In [34]:
output_files = {
    "CEAS-08":
        EMAIL_OUT / "ceas_08_clean.csv",

    "Enron":
        EMAIL_OUT / "enron_clean.csv",

    "Nazario":
        EMAIL_OUT / "nazario_clean.csv",

    "SpamAssassin":
        EMAIL_OUT / "spamassassin_clean.csv",

    "PhiUSIIL":
        WEBPAGE_OUT / "phiusiil_clean.csv",

    "UCI Phishing Websites":
        WEBPAGE_OUT / "uci_phishing_websites_clean.csv",

    "ISCX-URL2016 All":
        URL_OUT / "iscx_url2016_all_clean.csv",

    "ISCX-URL2016 Phishing":
        URL_OUT / "iscx_url2016_phishing_clean.csv",
}

for name, path in output_files.items():
    clean_data[name].to_csv(path, index=False)
    print(f"Exported {name}: {path.name}")

Exported CEAS-08: ceas_08_clean.csv
Exported Enron: enron_clean.csv
Exported Nazario: nazario_clean.csv
Exported SpamAssassin: spamassassin_clean.csv
Exported PhiUSIIL: phiusiil_clean.csv
Exported UCI Phishing Websites: uci_phishing_websites_clean.csv
Exported ISCX-URL2016 All: iscx_url2016_all_clean.csv
Exported ISCX-URL2016 Phishing: iscx_url2016_phishing_clean.csv


In [35]:
cleaning_log_df = pd.DataFrame(cleaning_log)

cleaning_log_df.to_csv(
    CLEANING_RESULTS_DIR / "cleaning_log.csv",
    index=False
)

validation_results.to_csv(
    CLEANING_RESULTS_DIR / "validation_results.csv",
    index=False
)

print("Cleaning log exported.")
print("Validation results exported.")

Cleaning log exported.
Validation results exported.


In [36]:
print("Cleaned datasets:")

for name, path in output_files.items():
    print(
        f"{name:26s} "
        f"{'OK' if path.exists() else 'MISSING'}"
    )

print("\nCleaning documentation:")

for filename in [
    "cleaning_log.csv",
    "validation_results.csv"
]:
    path = CLEANING_RESULTS_DIR / filename

    print(
        f"{filename:26s} "
        f"{'OK' if path.exists() else 'MISSING'}"
    )

Cleaned datasets:
CEAS-08                    OK
Enron                      OK
Nazario                    OK
SpamAssassin               OK
PhiUSIIL                   OK
UCI Phishing Websites      OK
ISCX-URL2016 All           OK
ISCX-URL2016 Phishing      OK

Cleaning documentation:
cleaning_log.csv           OK
validation_results.csv     OK


# 14. Cleaning Summary and Documentation

## Purpose

This notebook performed reproducible data cleaning for the datasets used in
the SE-BRL experimental pipeline. All cleaning decisions were based on findings
established during `01_data_audit.ipynb`, while the original raw datasets
remained unchanged.

The cleaning process was intentionally conservative. Operations were limited
to documented data-quality issues that could be resolved objectively without
introducing human-assigned labels, subjective corrections, or information
leakage.

## Cleaning Procedures

### Exact Duplicate Removal

Complete duplicate rows were removed from datasets where exact duplication was
identified:

- UCI Phishing Websites: 11,055 → 5,849 rows
- ISCX-URL2016 All: 36,707 → 26,953 rows
- ISCX-URL2016 Phishing: 15,367 → 14,823 rows

Before removal, predictor-level duplicate groups were checked for conflicting
labels. The UCI dataset contained 64 feature-identical groups associated with
different native labels. These conflicting observations were preserved because
resolving them would require an unsupported labeling decision.

Only complete row duplicates, including the native label, were removed.

### Missing Email Text

Missing `subject` and `body` values were replaced with empty strings while
preserving the corresponding records:

- CEAS-08: 28 values
- Enron: 198 values
- Nazario: 4 values
- SpamAssassin: 17 values

No email observation was removed because of missing text.

### ISCX Non-Finite Values

Following duplicate removal, the ISCX datasets contained missing and infinite
numeric values.

Infinite values were standardized to `NaN`:

- ISCX-URL2016 All: 10 infinite values converted
- ISCX-URL2016 Phishing: 9 infinite values converted

Existing missing values were not statistically imputed. Imputation is deferred
to leakage-safe preprocessing so that any learned parameters can be derived
only from appropriate development or training data.

Existing `-1` values were preserved and were not assumed to represent missing
data.

### UCI Representation Standardization

Values loaded from the UCI ARFF file were represented as byte strings such as
`b'-1'`, `b'0'`, and `b'1'`.

These representations were converted to integer values while preserving their
native meanings. The native `Result` label remains represented by `-1` and `1`.

### Extreme Email Lengths

Extreme email-body lengths were reviewed programmatically.

The maximum observed body lengths were:

- CEAS-08: 143,996 characters
- Enron: 228,353 characters
- Nazario: 4,599,644 characters
- SpamAssassin: 299,201 characters

No records were removed solely because of their length. Length alone was not
considered sufficient evidence of corruption. Any downstream truncation or
length restriction required by feature extraction or modeling is deferred to
preprocessing.

### Native Label Preservation

All source-native labels were preserved during cleaning.

No universal phishing label was created at this stage. In particular, spam and
phishing were not assumed to represent equivalent classes. Any target-label
mapping required for SE-BRL experiments will therefore be defined explicitly
during downstream experimental construction using documented source semantics.

## Post-Cleaning Validation

Nineteen automated validation checks were performed after cleaning.

The checks verified:

- removal of complete duplicate rows where required;
- absence of missing email `subject` and `body` values;
- absence of infinite numeric values in the cleaned ISCX datasets;
- successful conversion of UCI values to integer representation;
- preservation of non-empty datasets; and
- successful retention of data required for subsequent processing.

All 19 validation checks passed.

## Final Cleaned Dataset Sizes

| Dataset | Rows | Columns |
|---|---:|---:|
| CEAS-08 | 39,154 | 7 |
| Enron | 29,767 | 3 |
| Nazario | 1,565 | 7 |
| SpamAssassin | 5,809 | 7 |
| PhiUSIIL | 235,795 | 56 |
| UCI Phishing Websites | 5,849 | 31 |
| ISCX-URL2016 All | 26,953 | 80 |
| ISCX-URL2016 Phishing | 14,823 | 80 |

## Decisions Deferred to Preprocessing

The following operations were intentionally not performed during cleaning:

- statistical imputation of structured missing values;
- interpretation or replacement of source-defined `-1` values;
- removal of constant or low-variance features;
- feature scaling or normalization;
- model-dependent encoding;
- email truncation;
- harmonized phishing-target construction;
- leakage-aware grouping and partition assignment; and
- development/calibration/test splitting.

These operations will be addressed only at the appropriate downstream stage to
avoid data leakage and preserve the distinction between cleaning and
model-dependent preprocessing.

## Generated Artifacts

Cleaned intermediate datasets were written under:

`experiments/data/interim/`

The following cleaning documentation was written under:

`experiments/results/cleaning/`

- `cleaning_log.csv`
- `validation_results.csv`

The original files under `experiments/datasets-raw/` were not modified.

## Next Stage

The cleaned intermediate datasets will be used by
`03_preprocessing.ipynb`.

The preprocessing stage will establish consistent model-ready representations,
define leakage-aware grouping, address deferred structured missing values,
prepare source-supported target mappings, and implement the preprocessing
operations required before experimental partitioning and model training.